# W4A16 on Colab T4

1. **Runtime → Change runtime type → T4 GPU** (or GPU, then confirm it is a T4).
2. Run the cells in order.

Colab free often gives the same **NVIDIA T4 (`sm_75`)** this kernel is built for. Sessions disconnect; rebuild if the runtime resets. Use this for CUDA tests and a first bench. Publish README numbers from a dedicated T4 VM if you can.

Get the repo onto this machine with **one** of the next two cells: zip upload (works with uncommitted local work) or `git clone` (only after you push `csrc/` and `setup.py`).

In [ ]:
import torch

assert torch.cuda.is_available(), "Runtime → Change runtime type → T4 GPU, then rerun"
name = torch.cuda.get_device_name(0)
print(name)
print("cuda", torch.version.cuda, "pytorch", torch.__version__)
if "T4" not in name:
    print("Warning: this notebook targets T4 / sm_75; results will not match the project GPU.")

## Option A — upload a zip of the local repo

On your Mac: zip the project (include `csrc/`, `setup.py`, `w4a16/`, `tests/`, `scripts/`). In Colab, upload `repo.zip` with the file browser, then run:

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("/content/repo.zip")
assert zip_path.is_file(), "Upload repo.zip to /content/repo.zip first"
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall("/content/src")

candidates = list(Path("/content/src").rglob("setup.py"))
assert candidates, "zip must contain setup.py"
repo = candidates[0].parent
print("repo", repo)

## Option B — clone GitHub (skip if you used the zip)

Use this if GitHub `main` already has these TinyLlama generate changes. Otherwise upload a zip of the local repo (Option A) so Colab matches your laptop.

In [ ]:
from pathlib import Path

repo = Path("/content/w4a16-linear-kernel")
if not (repo / "setup.py").is_file():
    !git clone https://github.com/chris320211/w4a16-linear-kernel.git {repo}
print("repo", repo)

In [ ]:
import os
from pathlib import Path

if "repo" not in globals():
    raise RuntimeError("Run the zip cell or uncomment the clone cell so `repo` is set")

os.chdir(repo)
print("cwd", Path.cwd())
%pip install -q pytest
!python setup.py build_ext --inplace
!python -m pytest tests/ -q

Operator bench (sequential baselines, CUDA events). TinyLlama generate is the next section and downloads the model (~2 GB).

In [ ]:
!python scripts/benchmark.py --iters 20 --warmup 5

## TinyLlama end-to-end

Downloads `TinyLlama/TinyLlama-1.1B-Chat-v1.0`, replaces compatible `nn.Linear` layers, then generates text. FP16 vs fused is **quality** (quantization error), not kernel correctness.

Prefill (`M > 1`) uses the naive fused kernel; decode (`M = 1`) uses the fast kernel. First run is slow because of the model download.

In [ ]:
%pip install -q transformers sentencepiece accelerate

In [ ]:
!python scripts/quantize_tinyllama.py --device cuda --output checkpoints/tinyllama-w4a16.pt

In [ ]:
!python scripts/generate.py --mode fp16 --max-new-tokens 32

In [ ]:
!python scripts/generate.py --mode fused --checkpoint checkpoints/tinyllama-w4a16.pt --max-new-tokens 32